# Caminho de custo mínimo em grid — modelo de fluxo em rede

A ideia central: **não existe "próximo passo" no modelo.**

O solver vê os 24 arcos como 24 interruptores independentes e decide liga/desliga
em cada um. A conexão do caminho não é imposta — ela emerge das restrições de balanço.

## 1. Os dados de entrada

In [1]:
import numpy as np
import pyomo.environ as pyo
import matplotlib.pyplot as plt

grid = np.zeros((3, 3), dtype=int)
grid[1, 1] = 1
grid[0, 1] = 1
grid[0, 2] = 1

custo_passo = 1
ORIGEM  = (0, 0)
DESTINO = (2, 2)

linha, coluna = grid.shape
print(grid)

[[0 1 1]
 [0 1 0]
 [0 0 0]]


## 2. Vizinhos

Esse é o teu código, funcionando. Note que a lista já nasce vazia e só recebe
o que passa no teste — sem `.remove()`, sem bug de índice.

In [2]:
vizinhos = {}

for i in range(linha):
    for j in range(coluna):
        v = []
        for (li, lj) in [(i-1, j), (i+1, j), (i, j+1), (i, j-1)]:
            if 0 <= li < linha and 0 <= lj < coluna:
                v.append((li, lj))
        vizinhos[(i, j)] = v

for n, vs in vizinhos.items():
    print(n, '->', vs)

(0, 0) -> [(1, 0), (0, 1)]
(0, 1) -> [(1, 1), (0, 2), (0, 0)]
(0, 2) -> [(1, 2), (0, 1)]
(1, 0) -> [(0, 0), (2, 0), (1, 1)]
(1, 1) -> [(0, 1), (2, 1), (1, 2), (1, 0)]
(1, 2) -> [(0, 2), (2, 2), (1, 1)]
(2, 0) -> [(1, 0), (2, 1)]
(2, 1) -> [(1, 1), (2, 2), (2, 0)]
(2, 2) -> [(1, 2), (2, 1)]


## 3. Arcos e custos

Aqui virei a chave pra **custo de chegada**: `custo_passo + valor_da_célula_de_destino`.

Sua versão original somava origem + destino, e isso cobrava o dobro de toda célula
intermediária (ela paga uma vez ao ser destino de um arco, e de novo ao ser origem
do seguinte). Troque a linha comentada se quiser comparar.

In [3]:
arcos = {}

for n, vs in vizinhos.items():
    for m in vs:
        arcos[(n, m)] = custo_passo + int(grid[m])
        # alternativa (sua original, cobra em dobro):
        # arcos[(n, m)] = int(grid[n]) + custo_passo + int(grid[m])

print('total de arcos:', len(arcos))
for k in list(arcos)[:4]:
    print(k, '->', arcos[k])

total de arcos: 24
((0, 0), (1, 0)) -> 1
((0, 0), (0, 1)) -> 2
((0, 1), (1, 1)) -> 2
((0, 1), (0, 2)) -> 2


## 4. O vetor b

É só isso que define origem e destino. Não existe restrição especial pra eles —
o modelo é idêntico nas 9 células, muda só o número do lado direito.

| célula | b | leitura |
|---|---|---|
| origem | −1 | sai 1 a mais do que entra |
| destino | +1 | entra 1 a mais do que sai |
| resto | 0 | o que entra, sai |

**Confira sempre:** a soma de todos os b tem que dar zero. Se não der, o modelo
é infactível por construção.

In [4]:
b = {n: 0 for n in vizinhos}
b[ORIGEM]  = -1
b[DESTINO] = +1

print('soma dos b:', sum(b.values()), '  (tem que ser 0)')

soma dos b: 0   (tem que ser 0)


## 5. O modelo

**Atenção à armadilha do Pyomo:** ele *achata* tuplas aninhadas.
A chave `((0,0),(1,0))` vira `(0,0,1,0)` — uma tupla plana de 4 elementos, `dimen=4`.

Era nisso que seu `pyo.Param((model.linha, model.linha), ...)` estava batendo.
Em vez de brigar com o achatamento, trabalhamos com ele: arco = 4 índices planos.

In [5]:
b

{(0, 0): -1,
 (0, 1): 0,
 (0, 2): 0,
 (1, 0): 0,
 (1, 1): 0,
 (1, 2): 0,
 (2, 0): 0,
 (2, 1): 0,
 (2, 2): 1}

In [6]:
for (n,m),v in arcos.items():
    print(n, '->', m, ':', v)

(0, 0) -> (1, 0) : 1
(0, 0) -> (0, 1) : 2
(0, 1) -> (1, 1) : 2
(0, 1) -> (0, 2) : 2
(0, 1) -> (0, 0) : 1
(0, 2) -> (1, 2) : 1
(0, 2) -> (0, 1) : 2
(1, 0) -> (0, 0) : 1
(1, 0) -> (2, 0) : 1
(1, 0) -> (1, 1) : 2
(1, 1) -> (0, 1) : 2
(1, 1) -> (2, 1) : 1
(1, 1) -> (1, 2) : 1
(1, 1) -> (1, 0) : 1
(1, 2) -> (0, 2) : 2
(1, 2) -> (2, 2) : 1
(1, 2) -> (1, 1) : 2
(2, 0) -> (1, 0) : 1
(2, 0) -> (2, 1) : 1
(2, 1) -> (1, 1) : 2
(2, 1) -> (2, 2) : 1
(2, 1) -> (2, 0) : 1
(2, 2) -> (1, 2) : 1
(2, 2) -> (2, 1) : 1


In [7]:
model = pyo.ConcreteModel()

# N = os 9 nós, cada um uma tupla (i,j)  -> dimen=2
model.N = pyo.Set(initialize=list(vizinhos.keys()), dimen=2)

# A = os 24 arcos, cada um uma tupla (i,j,k,l) achatada -> dimen=4
model.A = pyo.Set(initialize=list(arcos.keys()), dimen=4)

# custo de cada arco (achata as chaves pra bater com o Set)
model.c = pyo.Param(model.A, initialize={(*n, *m): v for (n, m), v in arcos.items()})

model.b = pyo.Param(model.N, initialize=b)

# a variável de decisão: liga ou não cada arco
model.x = pyo.Var(model.A, domain=pyo.Binary)

print('nós:', len(model.N), ' arcos:', len(model.A))

nós: 9  arcos: 24


In [11]:
model.pprint()

2 Set Declarations
    A : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     4 :    Any :   24 : {(0, 0, 1, 0), (0, 0, 0, 1), (0, 1, 1, 1), (0, 1, 0, 2), (0, 1, 0, 0), (0, 2, 1, 2), (0, 2, 0, 1), (1, 0, 0, 0), (1, 0, 2, 0), (1, 0, 1, 1), (1, 1, 0, 1), (1, 1, 2, 1), (1, 1, 1, 2), (1, 1, 1, 0), (1, 2, 0, 2), (1, 2, 2, 2), (1, 2, 1, 1), (2, 0, 1, 0), (2, 0, 2, 1), (2, 1, 1, 1), (2, 1, 2, 2), (2, 1, 2, 0), (2, 2, 1, 2), (2, 2, 2, 1)}
    N : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     2 :    Any :    9 : {(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)}

2 Param Declarations
    b : Size=9, Index=N, Domain=Any, Default=None, Mutable=False
        Key    : Value
        (0, 0) :    -1
        (0, 1) :     0
        (0, 2) :     0
        (1, 0) :     0
        (1, 1) :     0
        (1, 2) :     0
        (2, 0) :     0
        (2, 1) :     0
        (2, 2

## 6. Função objetivo

Um somatório só, sobre **arcos**. `c` é número fixo, `x` é a variável.

Toda a geometria (quem é vizinho de quem, quanto vale a célula de chegada) já foi
resolvida no pré-processamento e está congelada dentro de `c`.

In [8]:
model.obj = pyo.Objective(
    expr=sum(model.c[a] * model.x[a] for a in model.A),
    sense=pyo.minimize,
)

## 7. A restrição de balanço

O coração do modelo. Uma equação **por nó**, mas somando sobre **arcos**.

$$\sum_{m \in V(n)} x_{m,n} \;-\; \sum_{m \in V(n)} x_{n,m} \;=\; b_n$$

Repare na simetria no código: **a mesma lista de vizinhos nas duas somas**, mudando
só a ordem dos índices.

- `m.x[k, l, i, j]` → o nó `(i,j)` está em **segundo** lugar → arco **chegando**
- `m.x[i, j, k, l]` → o nó `(i,j)` está em **primeiro** lugar → arco **saindo**

É só a posição na chave que distingue entrada de saída.

In [9]:
def balanco(m, i, j):
    entra = sum(m.x[k, l, i, j] for (k, l) in vizinhos[(i, j)])
    sai   = sum(m.x[i, j, k, l] for (k, l) in vizinhos[(i, j)])
    return entra - sai == m.b[i, j]

model.balanco = pyo.Constraint(model.N, rule=balanco)

# olha a equação do nó (1,0): 3 vizinhos -> 6 termos
model.balanco[1, 0].pprint()

{Member of balanco} : Size=9, Index=N, Active=True
    Key    : Lower : Body                                                                          : Upper : Active
    (1, 0) :   0.0 : x[0,0,1,0] + x[2,0,1,0] + x[1,1,1,0] - (x[1,0,0,0] + x[1,0,2,0] + x[1,0,1,1]) :   0.0 :   True


## 8. Resolver

In [10]:
res = pyo.SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')

print(res.solver.termination_condition)
print('custo total:', pyo.value(model.obj))

AttributeError: 'CPLEXSHELL' object has no attribute 'solver'

## 9. Ler a resposta

O solver devolve um **conjunto** de arcos ligados, sem ordem nenhuma.
Pra virar caminho, você monta um dicionário `de -> para` e caminha nele.

In [ ]:
ativos = [a for a in model.A if pyo.value(model.x[a]) > 0.5]
print('arcos ativos:', ativos)

proximo = {(a[0], a[1]): (a[2], a[3]) for a in ativos}

caminho, atual = [ORIGEM], ORIGEM
while atual != DESTINO:
    atual = proximo[atual]
    caminho.append(atual)

print('caminho:', caminho)

## 10. Visualizar

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(grid, cmap='Greys', vmin=0, vmax=2)

ys = [p[0] for p in caminho]
xs = [p[1] for p in caminho]
ax.plot(xs, ys, '-o', color='crimson', linewidth=2.5, markersize=9)

ax.plot(ORIGEM[1],  ORIGEM[0],  'o', color='seagreen', markersize=16, zorder=3)
ax.plot(DESTINO[1], DESTINO[0], 's', color='seagreen', markersize=16, zorder=3)

for i in range(linha):
    for j in range(coluna):
        ax.text(j, i, grid[i, j], ha='center', va='center',
                color='steelblue', fontsize=13)

ax.set_xticks(range(coluna)); ax.set_yticks(range(linha))
ax.set_title(f'custo = {pyo.value(model.obj):.0f}')
plt.show()

## Experimentos pra fixar

1. **Troque o destino** para `(0,2)` e rode de novo. Repare que você só mexeu
   no `b` — o modelo não mudou em nada.

2. **Sabote uma restrição.** Comente a linha do `model.balanco` e rode.
   O solver vai devolver custo 0 (não liga arco nenhum), porque sem balanço
   nada obriga a sair da origem.

3. **Ponha `custo_passo = 0`.** Agora existem infinitos ótimos empatados e o
   solver escolhe um arbitrariamente — pode até aparecer ciclo solto, já que
   dar volta passa a ser de graça.

4. **Grid maior com obstáculos.** Troque o valor 1 por 50 nas células bloqueadas
   e gere um grid 10×10 aleatório. O modelo não muda uma linha.

5. **Vizinhança-8.** Adicione as diagonais na lista de vizinhos — mas lembre de
   multiplicar o custo do passo diagonal por √2, senão o solver descobre que
   diagonal é atalho de graça.